# Cayley-Dickson Construction
This project is to represent the Cayley-Dickson construction for arbitrary base fields. <br>
When the base field is $\mathbb{R}$, this leads to the complex numbers, quaternions, octonions, etc.


In [8]:
import numpy as np

In [93]:
class Cayley:
    ''' A class to encode an array of numbers into a Cayley number.
        A CD_Alg object has the following attributes:
        coeff -  the array of coefficients as a numpy array. 
                
        field - the field of the coefficients
                'R' = real numbers, 0 = integers, p (prime)= finite field. 
                
        degree - the number of successive quadratic extensions needed to 
                represent the number. This is the smallest integer such 
                that 2^degree >= len(coeff). For example, the number 1
                has degree 0, the number 1 + i has degree 1, the number
                1 + i + j has degree 2, and so on. 
                
        real - the real part of the Cayley number.   '''
    
    def __init__(self, coeff=np.array([1]), degree=None, field='R'):
        if len(coeff) == 0: # edge case for empty array
            degree = 0
        if degree == None: # if degree not specified, use the smallest possible
            degree = (len(coeff) - 1).bit_length() # smallest extension needed
        if len(coeff) > (1 << degree): # check that it is large enough
            raise ValueError('Degree too small for number of coefficients')
        # pad with zeros
        coeff = np.pad(coeff, (0, (1 << degree) - len(coeff)), 'constant')

        self.coeff = coeff
        self.field = field
        self.degree = degree
        self.real = coeff[0]

    def __repr__(self):
        ''' written extremely badly! Definitely want to refactor this 
            and improve readability, but it works for now. '''
        def subscript(n):                
            # Base character 'i' in subscript Unicode
            base_char = chr(0x1D48A)
            
            # Unicode subscripts for digits 0-9
            subscripts = {
                '0': chr(0x2080),
                '1': chr(0x2081),
                '2': chr(0x2082),
                '3': chr(0x2083),
                '4': chr(0x2084),
                '5': chr(0x2085),
                '6': chr(0x2086),
                '7': chr(0x2087),
                '8': chr(0x2088),
                '9': chr(0x2089),
            }
            
            # Convert n into its subscript form
            subscript_digits = ''.join(subscripts[digit] for digit in str(n))
            
            # Return the result
            return base_char + subscript_digits
        non_zero_index = 0
        while self.coeff[non_zero_index] == 0:
            non_zero_index += 1
            if non_zero_index == len(self.coeff):
                return '0'
        if non_zero_index == 0:
            string = f'{self.coeff[non_zero_index]}' 
        else: 
            first_coeff = self.coeff[non_zero_index]
            if first_coeff == 1:
                first_coeff = ''
            elif first_coeff == -1:
                first_coeff = '-'
            string = f'{first_coeff}{subscript(non_zero_index)}'
        for digit in range(non_zero_index + 1, 1 << self.degree):
            if self.coeff[digit] != 0:
                string += f'{display(self.coeff[digit])}{subscript(digit)}'
        return string
    
    def left(self): # left half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return self
        left_coeff = self.coeff[:1 << (self.degree - 1)]
        left_degree = self.degree - 1
        return Cayley(left_coeff, left_degree, self.field)
    
    def right(self): # right half of coefficients as a Cayley number
        if self.degree == 0: # base case
            return Cayley([0], 0, self.field)
        right_coeff = self.coeff[1 << (self.degree - 1):]
        right_degree = self.degree - 1
        return Cayley(right_coeff, right_degree, self.field)

    def conj(self): # conjugate of a Cayley number (a, b)* = (a*, -b)
        if self.degree == 0: # base case
            return self
        else:
            if self.field in ['R', 'Z', 0]: # characteristic 0 case
                conj_coeff = self.coeff.copy() * -1 
                conj_coeff[0] *= -1
            else: # characteristic p case
                p = self.field
                conj_coeff = (self.coeff.copy() * -1) % p
                conj_coeff[0] = (conj_coeff[0] * -1) % p
            return Cayley(conj_coeff, self.degree, self.field)

    def __eq__(self, other):
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        eq_coeffs = (list(padded_self.coeff) == list(padded_self.coeff))
        eq_field = (padded_self.field == padded_self.field)
        return eq_coeffs and eq_field
    
    def __add__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        sum_coeff = np.add(padded_self.coeff, padded_other.coeff)
        return Cayley(sum_coeff, D, self.field)
    
    def __sub__(self, other):
        if self.field != other.field:
            raise ValueError('Fields must match')
        D = max(self.degree, other.degree) # make sure the degrees match
        padded_self = Cayley(self.coeff, D, self.field)
        padded_other = Cayley(other.coeff, D, other.field)
        diff_coeff = np.subtract(padded_self.coeff, padded_other.coeff)
        return Cayley(diff_coeff, D, self.field)
    
    def __mul__(self, other):
        if type(other) in [float, int]: # Scalar multiplication
            if type(other) == int and self.field not in ['R', 0]:
                p = self.field  # characteristic p case
                return Cayley((self.coeff * other) % p, field=self.field)
            return Cayley(self.coeff * other, field=self.field) # characteristic 0
        
        elif type(other) == Cayley: # Cayley number multiplication
            if self.field != other.field: # base fields should match
                raise ValueError('Fields must match to multiply')
            if self.degree == 0:    # base case is when the degree is 0
                return Cayley(other.coeff * self.coeff[0],
                                other.degree, self.field)
            elif other.degree == 0: 
                return Cayley(self.coeff * other.coeff[0],
                               self.degree, self.field)
            else: # break into smaller Cayley numbers for recursive multiplication
                D = max(self.degree, other.degree) # make sure the degrees match
                padded_self = Cayley(self.coeff, D, self.field)
                padded_other = Cayley(other.coeff, D, other.field)
                # multiplication formula is given by
                # (a_L, a_R)(b_L, b_R) = (a_L b_L - b_R* a_R, b_R a_L + a_R b_L* )
                a_L = Cayley(padded_self.left().coeff, D - 1, self.field)
                a_R = Cayley(padded_self.right().coeff, D - 1, self.field)
                b_L = Cayley(padded_other.left().coeff, D - 1, other.field)
                b_R = Cayley(padded_other.right().coeff, D - 1, other.field)

                prod_right = b_R * a_L + a_R * b_L.conj()  # recursive call 
                prod_left = a_L * b_L - b_R.conj() * a_R        
                product_coeff = np.concatenate((prod_left.coeff, prod_right.coeff))
                return Cayley(product_coeff, D, self.field)
        else:
            raise ValueError('Multiplication not defined for these types')
    
    def norm(self, as_scalar=True):
        ''' Returns the norm of a Cayley number N(x) = x.conj() * x'''
        if as_scalar:
            return (self.conj() * self).real
        else:
            return self.conj() * self
    
    def inv(self):
        ''' Returns the inverse of a Cayley number, if it exists. '''
        if self.norm() == 0:
            raise ValueError('Cannot invert a Cayley number with norm 0')

In [10]:
a =Cayley([0.1,1,0,-1,0,-3,-.1,0,0,1], degree=None, field='R')
def imag(n): # returns the cayley number with only i_n component
    length = n.bit_length()
    coeff = np.zeros(1 << length, dtype='int32')
    coeff[n] = 1
    return Cayley(coeff, length, field=0)

A = np.zeros((8,8), dtype='object')
for i in range(8):
    for j in range(8):
        A[i,j] = str(imag(i) * imag(j))

A

array([['1', '𝒊₁', '𝒊₂', '𝒊₃', '𝒊₄', '𝒊₅', '𝒊₆', '𝒊₇'],
       ['𝒊₁', '-1', '𝒊₃', '-𝒊₂', '𝒊₅', '-𝒊₄', '-𝒊₇', '𝒊₆'],
       ['𝒊₂', '-𝒊₃', '-1', '𝒊₁', '𝒊₆', '𝒊₇', '-𝒊₄', '-𝒊₅'],
       ['𝒊₃', '𝒊₂', '-𝒊₁', '-1', '𝒊₇', '-𝒊₆', '𝒊₅', '-𝒊₄'],
       ['𝒊₄', '-𝒊₅', '-𝒊₆', '-𝒊₇', '-1', '𝒊₁', '𝒊₂', '𝒊₃'],
       ['𝒊₅', '𝒊₄', '-𝒊₇', '𝒊₆', '-𝒊₁', '-1', '-𝒊₃', '𝒊₂'],
       ['𝒊₆', '𝒊₇', '𝒊₄', '-𝒊₅', '-𝒊₂', '𝒊₃', '-1', '-𝒊₁'],
       ['𝒊₇', '-𝒊₆', '𝒊₅', '𝒊₄', '-𝒊₃', '-𝒊₂', '𝒊₁', '-1']], dtype=object)

In [13]:
zero_divisor_1 = imag(1) + imag(10)
zero_divisor_2 = imag(5) + imag(14)
zero_divisor_3 = (imag(1) * -1) - imag(10) 

zero_divisor_1 * zero_divisor_3 

2

In [91]:
# idea for faster multiplication
def sgn(i, j):
    ''' e_ie_j = (-1) ^ sgn(i, j) e_(i XOR j) , where i > j'''
    if i == 0:
        return 0
    elif j == 0:
        return 0
    elif i == j:
        return 1
    elif j < i:
        return 1 - sgn(j, i)
    else:
        i_length = i.bit_length() # i = sum_{k < i_length} i_k 2^k
        j_length = j.bit_length()
        max_len = max(i_length, j_length)

        hamming_i = i.bit_count() # |\{i_k : i_k = 1\}|
        i_bits = {k : (i & (1 << k)) >> k for k in range(max_len)} # {k : i_k}

        upper_j = j >> i_length # bits of j above biggest bit of i
        hamming_upper_j = upper_j.bit_count()
        lower_j_bits = {l : j & (1 << l) for l in range(i_length)} 

        j_bits = {l : (j & (1 << l)) >> l for l in range(max_len)} # {l : j_l}

        

        # result = (hamming_i % 2) * (hamming_upper_j % 2)
        result = 0
        high_inx = 0
        while high_inx in j_bits: 
            for low_inx in range(high_inx + 1):
                if i_bits[low_inx] == 1 and j_bits[high_inx] == 1:
                    result = 1 - result
            high_inx += 1
        return 1 - result
            



In [96]:
for i in range(16, 32):
    for j in range(16, 32):    
        bool = (imag(i) * imag(j) == imag(i ^ j) * (-1) ** sgn(i, j))
        if not bool:
            print(f'Failed for i = {i}, j = {j}')
            print(f'LHS = {imag(i) * imag(j)}')
            print(f'RHS = {imag(i ^ j) * (-1) ** sgn(i, j)}')
            

KeyboardInterrupt: 

In [99]:
for i in range(1234,1244):
    for j in range(34543, 34553):
        print(sgn(i, j))

1
0
0
1
1
1
1
0
0
1
1
1
0
1
0
1
0
1
0
1
0
0
0
0
0
1
1
1
1
1
0
1
0
0
1
1
0
0
1
1
1
1
1
0
0
1
1
0
0
1
1
0
1
0
1
1
0
1
0
1
1
0
0
0
0
0
0
0
0
1
1
1
0
0
1
0
1
1
0
1
0
1
1
0
0
0
0
1
1
1
0
0
1
0
1
0
1
0
1
1


In [101]:
%timeit sgn(12344543234543, 345234565434565443)

622 µs ± 10.2 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
